# ML Lab — pierwszy eksperyment z regresją liniową

Przechodzimy przez eksperyment w kolejności z naszej rozmowy: poznanie danych,
podział, trening, predykcje, metryki, baseline, porównanie treningu z testem
oraz cross-validation. Na dole jest miejsce na samodzielny eksperyment z Ridge.

**Uruchamiaj komórki od góry do dołu.** Korzystaj z kernela środowiska projektu
`ml-lab` (interpreter `.venv/bin/python`). Zmienne utworzone w jednej komórce
są dostępne w kolejnych. Żeby sprawdzić całość od początku, zrestartuj kernel
i uruchom wszystkie komórki.

Celem jest przewidywanie ilościowego wskaźnika postępu choroby po roku
na podstawie 10 cech pacjenta. To ćwiczenie oceny modeli, a nie ocena ich
przydatności klinicznej.

## 1. Wczytanie danych — czym są X i y?

- `X` zawiera cechy wejściowe: 442 pacjentów i 10 cech.
- `y` zawiera rzeczywisty wynik każdego pacjenta, który chcemy przewidywać.
- `return_X_y=True` zwraca parę danych wejściowych i wartości docelowych.
- `as_frame=True` sprawia, że `X` jest tabelą pandas (`DataFrame`), a `y` kolumną (`Series`).

Nazwy `X` i `y` są konwencją. Wiersze obu obiektów odpowiadają tym samym pacjentom.
Cechy wczytywane domyślnie przez `load_diabetes` są już wycentrowane i przeskalowane;
nie przedstawiają surowych wartości pomiarów.

[Dokumentacja load_diabetes](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html)

In [1]:
from pandas import DataFrame, Series
from sklearn.datasets import load_diabetes

X, y = load_diabetes(return_X_y=True, as_frame=True)

print("Rozmiar X:", X.shape)
print("Rozmiar y:", y.shape)
X.head()

Rozmiar X: (442, 10)
Rozmiar y: (442,)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


## 2. Obejrzenie danych

`describe()` podsumowuje rozkład wartości docelowych. Skala `y` jest ważna
przy interpretacji MAE i RMSE: te metryki mają tę samą jednostkę co target.

In [2]:
y.describe()

count    442.000000
mean     152.133484
std       77.093005
min       25.000000
25%       87.000000
50%      140.500000
75%      211.500000
max      346.000000
Name: target, dtype: float64

In [3]:
y.head()

0    151.0
1     75.0
2    141.0
3    206.0
4    135.0
Name: target, dtype: float64

In [4]:
# Tymczasowo dołączamy target do cech, żeby obejrzeć je razem.
# assign tworzy nową tabelę — X nadal zawiera tylko 10 cech wejściowych.
X.assign(target=y).head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


## 3. Podział na trening i test

Model uczy się na `X_train` i `y_train`. Oceniamy go również na `X_test`
i `y_test`, których nie przekazujemy do treningu.

- `test_size=0.2` przeznacza około 20% obserwacji na test.
- `random_state=42` pozwala odtworzyć ten sam podział dla tych samych danych i ustawień.
  Liczba 42 nie ma specjalnych właściwości.
- Funkcja dzieli X i y wspólnie, zachowując powiązanie cech z wynikiem pacjenta.

[Dokumentacja train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Pacjenci w treningu:", len(X_train))
print("Pacjenci w teście:", len(X_test))

Pacjenci w treningu: 353
Pacjenci w teście: 89


## 4. Utworzenie i wytrenowanie LinearRegression

`LinearRegression()` tworzy obiekt modelu. Dopiero `fit(X_train, y_train)`
uczy go zależności między cechami a wynikiem. Funkcja zwraca wytrenowany model,
którego będziemy później używać do predykcji.

[Dokumentacja LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

In [6]:
from sklearn.linear_model import LinearRegression


def train_regression(X_train: DataFrame, y_train: Series) -> LinearRegression:
    reg = LinearRegression()
    reg.fit(X_train, y_train)
    return reg


reg = train_regression(X_train, y_train)

## 5. Predykcje na danych testowych

`predict(X_test)` oblicza przewidywania dla pacjentów testowych.
Nie przekazujemy tu `y_test`: to prawdziwe odpowiedzi, z którymi dopiero
porównamy przewidywania. Nie trenujemy modelu ponownie.

In [7]:
y_pred = reg.predict(X_test)

print("Liczba predykcji:", len(y_pred))
print("Pierwsze 10 predykcji:", y_pred[:10])

Liczba predykcji: 89
Pierwsze 10 predykcji: [139.5475584  179.51720835 134.03875572 291.41702925 123.78965872
  92.1723465  258.23238899 181.33732057  90.22411311 108.63375858]


## 6. Ocena — MAE, RMSE i R²

Każdej funkcji metryki przekazujemy najpierw prawdziwe wyniki, a następnie predykcje.

- **MAE** to średnia bezwzględnych błędów. Mniejsze jest lepsze.
- **RMSE** to pierwiastek ze średniej kwadratów błędów. Mocniej reaguje na duże pomyłki.
  Mniejsze jest lepsze.
- **R²** odnosi sumę kwadratów błędów do przewidywania średniej z ocenianego zbioru.
  Wynik 1 oznacza idealne przewidywania, 0 wynik równy tej średniej, a wartość ujemna
  wynik gorszy. R² nie jest procentem trafnych odpowiedzi.

Funkcja `calculate_metrics` zbiera te same trzy obliczenia w jednym miejscu,
żeby używać ich później dla obu modeli i obu zbiorów.

[Dokumentacja metryk regresji](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)

In [8]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score


def calculate_metrics(y_true, y_pred) -> dict[str, float]:
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R²": r2_score(y_true, y_pred),
    }


regression_test_metrics = calculate_metrics(y_test, y_pred)
Series(regression_test_metrics, name="Linear Regression — test").round(4)

MAE     42.7941
RMSE    53.8534
R²       0.4526
Name: Linear Regression — test, dtype: float64

## 7. Punkt odniesienia — baseline przewidujący średnią

Sam błąd około 43 jednostek nie mówi, ile daje model. Porównujemy go z prostym
baseline'em: `DummyRegressor(strategy="mean")` przewiduje każdemu pacjentowi
średnią z `y_train`. Średnią wyznaczamy podczas treningu, bez używania `y_test`.

Osobna funkcja trenuje baseline i zwraca jego obiekt. Jego predykcje zapisujemy
w `y_pred_baseline`, zachowując zarówno model, jak i przewidywania regresji.

[Dokumentacja DummyRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html)

In [9]:
from sklearn.dummy import DummyRegressor


def train_baseline(X_train: DataFrame, y_train: Series) -> DummyRegressor:
    baseline = DummyRegressor(strategy="mean")
    baseline.fit(X_train, y_train)
    return baseline


baseline = train_baseline(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)
baseline_test_metrics = calculate_metrics(y_test, y_pred_baseline)

print("Średnia z y_train:", y_train.mean())
DataFrame({
    "Linear Regression — test": regression_test_metrics,
    "Baseline — test": baseline_test_metrics,
}).round(4)

Średnia z y_train: 153.73654390934846


,Linear Regression — test,Baseline — test
MAE,42.7941,64.0065
RMSE,53.8534,73.2225
R²,0.4526,-0.0120


Regresja obniża MAE względem przewidywania samej średniej o około 33% na tym teście.
Baseline może mieć lekko ujemne R²: przewiduje średnią z treningu, a wzór R²
odnosi się do średniej z ocenianego zbioru testowego. Te średnie mogą się różnić.

## 8. Porównanie dla pojedynczych pacjentów

Zestawiamy prawdziwy wynik, predykcje i błędy dla pierwszych 10 pacjentów testowych.
Indeks tabeli pochodzi z oryginalnych danych; nie jest kolejnym numerem w teście.

`Error = Actual - Predicted`: dodatnia wartość oznacza, że model zaniżył wynik,
a ujemna, że go zawyżył. Metryka MAE używa wartości bezwzględnych tych błędów.
Regresja nie musi wygrywać z baseline'em dla każdego pacjenta, żeby być lepsza średnio.

In [10]:
results = DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Baseline": y_pred_baseline,
})
results["Error"] = results["Actual"] - results["Predicted"]
results["Baseline error"] = results["Actual"] - results["Baseline"]

results.head(10).round(2)

,Actual,Predicted,Baseline,Error,Baseline error
287,219.0,139.55,153.74,79.45,65.26
211,70.0,179.52,153.74,-109.52,-83.74
72,202.0,134.04,153.74,67.96,48.26
321,230.0,291.42,153.74,-61.42,76.26
73,111.0,123.79,153.74,-12.79,-42.74
418,84.0,92.17,153.74,-8.17,-69.74
367,242.0,258.23,153.74,-16.23,88.26
354,272.0,181.34,153.74,90.66,118.26
281,94.0,90.22,153.74,3.78,-59.74
148,96.0,108.63,153.74,-12.63,-57.74


## 9. Porównanie treningu z testem — czy widać przeuczenie?

Używamy tych samych wytrenowanych obiektów i wykonujemy predykcje także dla `X_train`.
Nie wywołujemy ponownie `fit`.

Dużo lepsze wyniki treningowe niż walidacyjne lub testowe mogą sugerować przeuczenie.
Podobne błędy na obu zbiorach nie dowodzą automatycznie niedouczenia ani nie wykluczają
przeuczenia we wszystkich innych podziałach. Są wskazówką do dalszej oceny.

In [11]:
y_pred_train = reg.predict(X_train)
y_pred_baseline_train = baseline.predict(X_train)

regression_train_metrics = calculate_metrics(y_train, y_pred_train)
baseline_train_metrics = calculate_metrics(y_train, y_pred_baseline_train)

regression_metrics = DataFrame({
    "Trening": regression_train_metrics,
    "Test": regression_test_metrics,
})
baseline_metrics = DataFrame({
    "Trening": baseline_train_metrics,
    "Test": baseline_test_metrics,
})

print("Linear Regression:")
print(regression_metrics.round(4).to_string())
print("\nBaseline (średnia z y_train):")
print(baseline_metrics.round(4).to_string())

Linear Regression:
      Trening     Test
MAE   43.4835  42.7941
RMSE  53.5588  53.8534
R²     0.5279   0.4526

Baseline (średnia z y_train):
      Trening     Test
MAE   66.4482  64.0065
RMSE  77.9513  73.2225
R²     0.0000  -0.0120


MAE i RMSE regresji są bardzo podobne w treningu i teście. Na tym podziale
nie widać silnego przeuczenia. Niższe testowe R² przy podobnym RMSE jest możliwe,
ponieważ R² zależy również od zmienności targetu w ocenianym zbiorze.

## 10. Cross-validation — ocena na pięciu częściach danych

Jeden wynik zależy od podziału. W 5-fold cross-validation dzielimy **wyłącznie dane
treningowe** na pięć części. W każdym przebiegu świeży model uczy się na czterech
częściach, a piąta służy do walidacji. Każda część raz jest walidacyjna.

`cross_val_score` wykonuje treningi i ocenę za nas. Przekazujemy nowy
`LinearRegression()`, bez wcześniejszego `fit`. Nie zmieniamy wytrenowanego `reg`.
Dla regresji `cv=5` korzysta z KFold bez dodatkowego mieszania; wcześniejszy
`train_test_split` już przemieszał dane. To pięć foldów jednego podziału,
a nie pięć niezależnych losowań całego eksperymentu.

`neg_mean_absolute_error` zwraca ujemne MAE, bo system scoringu traktuje większą
wartość jako lepszą. Zmieniamy znak tablicy, żeby otrzymać zwykłe dodatnie MAE.

[Dokumentacja cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)

In [12]:
from sklearn.model_selection import cross_val_score

cross_val_scores = cross_val_score(
    LinearRegression(),
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error",
)

print("Surowe wyniki scoringu:", cross_val_scores)

Surowe wyniki scoringu: [-41.09713505 -49.26831278 -43.21638998 -45.52066043 -48.6141468 ]


In [13]:
# Minus zmienia znak każdego elementu tablicy NumPy — pętla nie jest potrzebna.
mae_folds = -cross_val_scores

print("MAE dla każdego foldu:", mae_folds)
print("Średnie MAE:", mae_folds.mean())
print("Odchylenie standardowe MAE:", mae_folds.std())

MAE dla każdego foldu: [41.09713505 49.26831278 43.21638998 45.52066043 48.6141468 ]
Średnie MAE: 45.54332900504237
Odchylenie standardowe MAE: 3.114136341802173


Średnie MAE wynosi około 45,54, a odchylenie między foldami około 3,11.
Wyniki są dość zbliżone w tych pięciu częściach danych. Odchylenie opisuje rozrzut
pięciu wartości MAE; nie jest średnim błędem pacjenta ani przedziałem ufności.

Wynik na teście już oglądaliśmy. Kolejne decyzje o modelach i ich ustawieniach
opieramy na cross-validation na danych treningowych, zamiast dobierać ustawienia
pod `X_test` i `y_test`.

## 11. Twój następny eksperyment — Ridge

Tutaj możesz dopisać własny kod. Ridge dodaje do regresji liniowej karę za duże
współczynniki, a `alpha` określa siłę regularyzacji. Nie zakładamy z góry,
że poprawi to wynik.

1. Zaimportuj `Ridge` z `sklearn.linear_model`.
2. Oceń `Ridge(alpha=1.0)` przez `cross_val_score`, używając tych samych `X_train`,
   `y_train`, `cv=5` i `scoring="neg_mean_absolute_error"`.
3. Zmień znak wyników, wypisz pięć MAE, średnią i odchylenie. Porównaj je
   z `mae_folds` dla zwykłej regresji. Zachowaj osobne zmienne dla Ridge.

Czy średnie MAE spadnie względem około 45,54? Na razie korzystaj tylko z walidacji
na danych treningowych, bez kolejnego sprawdzania testu.

[Dokumentacja Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)